In [1]:
!pip install datasets trimesh rtree torch_geometric


In [2]:
import torch

# Sistemdeki mevcut PyTorch ve CUDA sürümlerini tespit et
torch_version = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda.replace('.', '') # Örn: '12.1' -> '121'

# Özel tekerlek (wheel) linkini dinamik olarak oluştur
url = f"https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version}.html"

print(f"Sistem Tespiti: PyTorch {torch_version} | CUDA {cuda_version}")
print(f"İndirme Linki: {url}\n")
print("Kurulum başlatılıyor...\n")

# Kurulumu dinamik link ile tetikle
!pip install torch_cluster -f {url}

Sistem Tespiti: PyTorch 2.10.0 | CUDA 128
İndirme Linki: https://data.pyg.org/whl/torch-2.10.0+cu128.html

Kurulum başlatılıyor...

Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html


In [3]:
import os
import json
import numpy as np
import torch
import trimesh
from torch_geometric.data import Data, Dataset
from datasets import load_dataset
from huggingface_hub import hf_hub_download
import shutil

class PartNeXtDataset(Dataset):
    def __init__(self, mesh_root, hf_repo="AuWang/PartNeXt", split='train', num_points=2048, transform=None):
        """
        mesh_root: Artık kalıcı bir dizin değil, geçici (kullan-at) dosyaların inip silineceği klasör.
        hf_repo: HuggingFace'teki annotasyon veri seti yolu.
        """
        super().__init__(mesh_root, transform)
        self.mesh_root = mesh_root
        self.num_points = num_points
        
        print(f"HuggingFace üzerinden '{split}' veri seti yükleniyor...")
        # HuggingFace datasets kütüphanesi ile arrow/parquet formatındaki etiketleri çekiyoruz
        self.dataset = load_dataset(hf_repo, split=split)
        
        # Geçici indirme klasörünü oluştur
        os.makedirs(self.mesh_root, exist_ok=True)
        
    def len(self):
        return len(self.dataset)
        
    def _parse_hierarchy(self, node, mask_to_label):
        """
        Ağaç yapısındaki hierarchyList'i gezerek yaprak düğümlerdeki (leaf node)
        maskId değerlerini, evrensel sınıf ID'si olan refNodeId'ye eşitler.
        """
        if "children" in node and len(node["children"]) > 0:
            for child in node["children"]:
                self._parse_hierarchy(child, mask_to_label)
        else:
            if "maskId" in node:
                # refNodeId: O parçanın küresel şablondaki (template) ID'si
                mask_to_label[int(node["maskId"])] = int(node.get("refNodeId", 0))

    def get(self, idx):
        row = self.dataset[idx]
        
        model_id = row['model_id']
        type_id = row['type_id']
        
        # HuggingFace üzerinden arrow formatında gelen veriler string olarak gelir, parse ediyoruz.
        masks = json.loads(row['masks']) if isinstance(row['masks'], str) else row['masks']
        hierarchy = json.loads(row['hierarchyList']) if isinstance(row['hierarchyList'], str) else row['hierarchyList']
        
        # 1. Hiyerarşiden "Mask ID -> Sınıf ID (refNodeId)" haritasını çıkar
        mask_to_label = {}
        for root_node in hierarchy:
            self._parse_hierarchy(root_node, mask_to_label)
            
        # Nesne kategorisini (Object Category) ağacın en tepesindeki ana düğümden al
        root_category_id = int(hierarchy[0].get("refNodeId", 0)) if hierarchy else 0

        # ---------------------------------------------------------
        # KULLAN-AT İNDİRME SİSTEMİ (Diski korur)
        # ---------------------------------------------------------
        file_path_in_repo = f"glbs/{type_id}/{model_id}.glb"
        local_glb_path = ""
        
        try:
            # Sadece ilgili 3D modeli indir ve önbelleği (symlink) kapat ki diskte yer tutmasın
            local_glb_path = hf_hub_download(
                repo_id="AuWang/PartNeXt_mesh", 
                repo_type="dataset", 
                filename=file_path_in_repo, 
                local_dir=self.mesh_root,
                local_dir_use_symlinks=False
            )
            
            # 2. GLB Dosyasını Yükle (scene olarak yükleyip mesh'leri ayırıyoruz)
            scene = trimesh.load(local_glb_path, force='scene')
            meshes = scene.dump(concatenate=False)
            
            # 3. Her mesh için yüzey (face) etiket matrislerini -1 (boş) ile başlat
            mesh_face_labels = {str(i): np.full(len(m.faces), -1, dtype=np.int64) for i, m in enumerate(meshes)}
            
            # 4. Maskeleri JSON'dan okuyup mesh yüzeylerine uygula
            for mask_id_str, mask_dict in masks.items():
                mask_id = int(mask_id_str)
                if mask_id not in mask_to_label:
                    continue
                
                semantic_label = mask_to_label[mask_id]
                
                for mesh_idx_str, face_list in mask_dict.items():
                    if mesh_idx_str in mesh_face_labels:
                        # Belirtilen yüzeylere anlamsal sınıfı ata
                        mesh_face_labels[mesh_idx_str][face_list] = semantic_label
                        
            # 5. Point Cloud Örnekleme (Mesh yüzey alanına orantılı)
            total_area = sum([m.area for m in meshes])
            all_points = []
            all_labels = []
            
            for i, m in enumerate(meshes):
                if total_area == 0 or m.area == 0:
                    continue
                    
                # O anki mesh'in alanına göre 2048 noktanın kaçının buradan alınacağını hesapla
                num_samples = int(self.num_points * (m.area / total_area))
                if num_samples <= 0:
                    continue
                    
                # Yüzeyden noktaları rastgele örnekle (points) ve noktanın düştüğü yüzeyi al (face_indices)
                points, face_indices = trimesh.sample.sample_surface(m, num_samples)
                
                # Yüzeyin etiketini, örneklendiği noktanın etiketi olarak kullan
                labels = mesh_face_labels[str(i)][face_indices]
                
                all_points.append(points)
                all_labels.append(labels)
                
            # Float hesaplamalarındaki tolerans kayıplarından dolayı nokta sayısını tam 2048'e sabitle (Padding/Truncating)
            if len(all_points) > 0:
                all_points = np.vstack(all_points)
                all_labels = np.concatenate(all_labels)
            else:
                all_points = np.zeros((self.num_points, 3))
                all_labels = np.full(self.num_points, -1)

            curr_points = all_points.shape[0]
            if curr_points < self.num_points:
                pad_indices = np.random.choice(curr_points, self.num_points - curr_points, replace=True)
                all_points = np.vstack([all_points, all_points[pad_indices]])
                all_labels = np.concatenate([all_labels, all_labels[pad_indices]])
            elif curr_points > self.num_points:
                choice = np.random.choice(curr_points, self.num_points, replace=False)
                all_points = all_points[choice]
                all_labels = all_labels[choice]
                
            # Etiketi olmayan kısımları (örn: modelde maskelenmemiş iç yüzeyler) 0 (arkaplan) sınıfına ata
            all_labels[all_labels == -1] = 0 
            
            # Mimarine girecek nihai Data nesnesi
            data = Data(
                pos=torch.tensor(all_points, dtype=torch.float),
                y=torch.tensor(all_labels, dtype=torch.long),
                category=torch.tensor([root_category_id], dtype=torch.long)
            )
            
        except Exception as e:
            # HuggingFace'te dosya yoksa, bağlantı koparsa veya dosya bozuksa sistemi çökertme.
            data = Data(
                pos=torch.zeros(self.num_points, 3),
                y=torch.zeros(self.num_points, dtype=torch.long),
                category=torch.tensor([root_category_id], dtype=torch.long)
            )
            
        finally:
            # KRİTİK ADIM: İşlem başarılı olsa da hata alsa da, diskte yer kaplamaması için 3D dosyayı sil.
            if local_glb_path and os.path.exists(local_glb_path):
                try:
                    os.remove(local_glb_path)
                except OSError:
                    pass
        
        if self.transform:
            data = self.transform(data)
            
        return data

In [4]:
def load_partnext_mappings(cache_dir="./data/PartNeXt"):
    meta_file = os.path.join(cache_dir, "partnext_meta_cache.json")
    
    # 1. Önbellekten (cache) oku
    if os.path.exists(meta_file):
        print("Kategori haritaları önbellekten (cache) yükleniyor...")
        with open(meta_file, "r") as f:
            data = json.load(f)
        cat_to_parts = {int(k): v for k, v in data["cat_to_parts"].items()}
        return data["cat_names"], cat_to_parts, data["num_categories"], data["total_parts"]

    print("Kategori ve Parça haritası HuggingFace'den çıkarılıyor (Klasörler değil, hiyerarşi okunuyor)...")
    from datasets import load_dataset
    dataset = load_dataset("AuWang/PartNeXt", split="train")
    
    cat_names_dict = {}
    cat_to_parts_raw = {}
    
    def extract_parts(node, valid_parts_set):
        if "refNodeId" in node:
            valid_parts_set.add(int(node["refNodeId"]))
        if "children" in node:
            for child in node["children"]:
                extract_parts(child, valid_parts_set)

    for row in tqdm(dataset, desc="Kategoriler Çıkarılıyor"):
        hierarchy = json.loads(row['hierarchyList']) if isinstance(row['hierarchyList'], str) else row['hierarchyList']
        if not hierarchy:
            continue
            
        # Kategori adını ve ID'sini ROOT (En tepe) düğümden alıyoruz! (type_id klasöründen değil)
        root_node = hierarchy[0]
        root_id = int(root_node.get("refNodeId", 0))
        root_name = root_node.get("name", f"Bilinmeyen_Kategori_{root_id}")
        
        cat_names_dict[root_id] = root_name
        
        if root_id not in cat_to_parts_raw:
            cat_to_parts_raw[root_id] = set()
            
        extract_parts(root_node, cat_to_parts_raw[root_id])

    # Kategori ID'leri sırayla gitmeyebilir, aradaki boşlukları doldurarak liste oluştur
    max_cat_id = max(cat_names_dict.keys())
    cat_names = [f"Bos_Kategori_{i}" for i in range(max_cat_id + 1)]
    for k, v in cat_names_dict.items():
        cat_names[k] = v

    cat_to_parts = {k: list(v) for k, v in cat_to_parts_raw.items()}
    num_categories = max_cat_id + 1
    total_parts = max([max(parts) for parts in cat_to_parts.values() if parts]) + 1

    os.makedirs(cache_dir, exist_ok=True)
    with open(meta_file, "w") as f:
        json.dump({
            "cat_names": cat_names,
            "cat_to_parts": cat_to_parts,
            "num_categories": num_categories,
            "total_parts": total_parts
        }, f)

    return cat_names, cat_to_parts, num_categories, total_parts

In [5]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [6]:
import os
import time
import math
import json
import random
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import to_dense_batch
from torch_cluster import fps, radius_graph, knn
import torch.compiler
from torch.amp import autocast, GradScaler

import triton
import triton.language as tl

torch._dynamo.config.capture_scalar_outputs = True

# -----------------------------------------------------------------------------
# TRITON CLIFFORD KERNEL
# -----------------------------------------------------------------------------
@triton.jit
def _clifford_prod_fwd_kernel(a_ptr, b_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    
    a0 = tl.load(a_ptr + offsets * 8 + 0, mask=mask, other=0.0)
    a1 = tl.load(a_ptr + offsets * 8 + 1, mask=mask, other=0.0)
    a2 = tl.load(a_ptr + offsets * 8 + 2, mask=mask, other=0.0)
    a3 = tl.load(a_ptr + offsets * 8 + 3, mask=mask, other=0.0)
    a4 = tl.load(a_ptr + offsets * 8 + 4, mask=mask, other=0.0)
    a5 = tl.load(a_ptr + offsets * 8 + 5, mask=mask, other=0.0)
    a6 = tl.load(a_ptr + offsets * 8 + 6, mask=mask, other=0.0)
    a7 = tl.load(a_ptr + offsets * 8 + 7, mask=mask, other=0.0)

    b0 = tl.load(b_ptr + offsets * 8 + 0, mask=mask, other=0.0)
    b1 = tl.load(b_ptr + offsets * 8 + 1, mask=mask, other=0.0)
    b2 = tl.load(b_ptr + offsets * 8 + 2, mask=mask, other=0.0)
    b3 = tl.load(b_ptr + offsets * 8 + 3, mask=mask, other=0.0)
    b4 = tl.load(b_ptr + offsets * 8 + 4, mask=mask, other=0.0)
    b5 = tl.load(b_ptr + offsets * 8 + 5, mask=mask, other=0.0)
    b6 = tl.load(b_ptr + offsets * 8 + 6, mask=mask, other=0.0)
    b7 = tl.load(b_ptr + offsets * 8 + 7, mask=mask, other=0.0)
    
    res0 = a0*b0 + a1*b1 + a2*b2 + a3*b3 - a4*b4 - a5*b5 - a6*b6 - a7*b7
    res1 = a0*b1 + a1*b0 - a2*b4 + a3*b6 + a4*b2 - a5*b7 - a6*b3 - a7*b5
    res2 = a0*b2 + a1*b4 + a2*b0 - a3*b5 - a4*b1 + a5*b3 - a6*b7 - a7*b6
    res3 = a0*b3 - a1*b6 + a2*b5 + a3*b0 + a4*b7 - a5*b2 + a6*b1 - a7*b4
    res4 = a0*b4 + a1*b2 - a2*b1 + a3*b7 + a4*b0 - a5*b6 + a6*b5 - a7*b3
    res5 = a0*b5 + a1*b7 + a2*b3 - a3*b2 + a4*b6 + a5*b0 - a6*b4 - a7*b1
    res6 = a0*b6 - a1*b3 + a2*b7 + a3*b1 - a4*b5 + a5*b4 + a6*b0 - a7*b2
    res7 = a0*b7 + a1*b5 + a2*b6 + a3*b4 + a4*b3 + a5*b1 + a6*b2 + a7*b0
    
    tl.store(out_ptr + offsets * 8 + 0, res0, mask=mask)
    tl.store(out_ptr + offsets * 8 + 1, res1, mask=mask)
    tl.store(out_ptr + offsets * 8 + 2, res2, mask=mask)
    tl.store(out_ptr + offsets * 8 + 3, res3, mask=mask)
    tl.store(out_ptr + offsets * 8 + 4, res4, mask=mask)
    tl.store(out_ptr + offsets * 8 + 5, res5, mask=mask)
    tl.store(out_ptr + offsets * 8 + 6, res6, mask=mask)
    tl.store(out_ptr + offsets * 8 + 7, res7, mask=mask)

class CliffordProductFunc(torch.autograd.Function):
    @staticmethod
    def forward(ctx, a, b):
        a = a.contiguous()
        b = b.contiguous()
        ctx.save_for_backward(a, b)
        out = torch.empty_like(a)
        n_elements = a.numel() // 8
        grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
        _clifford_prod_fwd_kernel[grid](a, b, out, n_elements, BLOCK_SIZE=1024)
        return out

    @staticmethod
    def backward(ctx, grad_output):
        a, b = ctx.saved_tensors
        grad_output = grad_output.contiguous()
        grad_a = grad_b = None
        rev_mask = torch.tensor([1, 1, 1, 1, -1, -1, -1, -1], device=a.device, dtype=a.dtype)
        
        if ctx.needs_input_grad[0]:
            b_rev = (b * rev_mask).contiguous()
            grad_a = triton_clifford_product(grad_output, b_rev)
        if ctx.needs_input_grad[1]:
            a_rev = (a * rev_mask).contiguous()
            grad_b = triton_clifford_product(a_rev, grad_output)
        return grad_a, grad_b

def triton_clifford_product(a, b):
    a_exp, b_exp = torch.broadcast_tensors(a, b)
    return CliffordProductFunc.apply(a_exp, b_exp)

@torch.compiler.disable
def safe_fps(*args, **kwargs):
    return fps(*args, **kwargs)

@torch.compiler.disable
def safe_radius_graph(*args, **kwargs):
    return radius_graph(*args, **kwargs)

@torch.compiler.disable
def safe_knn(*args, **kwargs):
    return knn(*args, **kwargs)


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class FixedPointsDeterministic:
    def __init__(self, num: int):
        self.num = num
    def __call__(self, data: Data) -> Data:
        num_nodes = data.num_nodes
        if num_nodes >= self.num:
            choice = torch.randperm(num_nodes)[:self.num]
        else:
            extra = torch.randint(0, num_nodes, (self.num - num_nodes,))
            choice = torch.cat([torch.randperm(num_nodes), extra], dim=0)
        for key, item in data:
            if torch.is_tensor(item) and item.size(0) == num_nodes:
                data[key] = item[choice]
        return data

class NormalizeUnitSphere:
    def __call__(self, data: Data) -> Data:
        pos = data.pos
        pos = pos - pos.mean(dim=0, keepdim=True)
        scale = pos.norm(dim=1).max().clamp(min=1e-6)
        data.pos = pos / scale
        return data

def precompute_fps_indices(data: Data, ratio1: float = 0.5, ratio2: float = 0.25) -> Data:
    pos = data.pos
    batch = torch.zeros(pos.size(0), dtype=torch.long, device=pos.device)
    data.fps_idx_1 = safe_fps(pos, batch, ratio=ratio1).cpu().long()
    
    pos2 = pos[data.fps_idx_1]
    batch2 = torch.zeros(pos2.size(0), dtype=torch.long, device=pos2.device)
    data.fps_idx_2 = safe_fps(pos2, batch2, ratio=ratio2).cpu().long()
    return data


def materialize_split(cache_root: str, mesh_root: str, split: str, num_points: int, seed: int, force_rebuild: bool = False):
    """
    cache_root: İşlenmiş PT (PyTorch) dosyalarının kaydedileceği klasör (Örn: './data/PartNeXt')
    mesh_root: 3D GLB dosyalarının bulunduğu klasör (Örn: './mesh_root/glbs')
    """
    cache_file = os.path.join(cache_root, f"partnext_{split}_n{num_points}_seed{seed}_fps.pt")
    if os.path.exists(cache_file) and not force_rebuild:
        print(f"[{time.strftime('%H:%M:%S')}] Cache bulundu: {cache_file}")
        return torch.load(cache_file, weights_only=False)

    print(f"[{time.strftime('%H:%M:%S')}] PartNeXt {split} split materyalize ediliyor...")
    seed_everything(seed)
    
    # DİKKAT: FixedPointsDeterministic çıkarıldı. Dataset zaten tam num_points kadar örnekliyor.
    transform = lambda data: precompute_fps_indices(NormalizeUnitSphere()(data))

    # Yeni veri seti sınıfımızı güncel parametrelerle çağırıyoruz
    dataset = PartNeXtDataset(
        mesh_root=mesh_root, 
        hf_repo="AuWang/PartNeXt", 
        split=split, 
        num_points=num_points, 
        transform=transform
    )

    materialized = []
    # Dataset üzerinden döngüye girip ön işlemleri yapıyoruz
    for i in range(len(dataset)):
        item = dataset[i].clone()
        item = Data(**{k: v.clone() if torch.is_tensor(v) else v for k, v in item})
        materialized.append(item)
        
        if (i + 1) % 1000 == 0 or (i + 1) == len(dataset):
            print(f"  {split}: {i + 1}/{len(dataset)}")

    os.makedirs(cache_root, exist_ok=True)
    torch.save(materialized, cache_file)
    return materialized

# -----------------------------------------------------------------------------
# EAGER GRAPH PREPARATION
# -----------------------------------------------------------------------------
@torch.no_grad()
def prepare_batch_data(data):
    assert data.pos.device.type == 'cuda', f"KRİTİK HATA: Veri CPU'da kaldı! Mevcut cihaz: {data.pos.device}"
    pos, batch = data.pos, data.batch
    num_nodes = torch.bincount(batch)
    offset = torch.cumsum(num_nodes, dim=0) - num_nodes
    fps_idx_1 = (data.fps_idx_1.view(num_nodes.size(0), -1) + offset.view(-1, 1)).reshape(-1)

    pos2, batch2 = pos[fps_idx_1], batch[fps_idx_1]

    num_nodes_2 = torch.bincount(batch2)
    fps_idx_2 = (data.fps_idx_2.view(num_nodes_2.size(0), -1) + (torch.cumsum(num_nodes_2, dim=0) - num_nodes_2).view(-1, 1)).reshape(-1)

    pos3, batch3 = pos2[fps_idx_2], batch2[fps_idx_2]

    # PartNeXt için radius değerleri nesne ölçeğine göre ince ayar isteyebilir
    edge_index_1 = safe_radius_graph(pos, r=0.10, batch=batch, max_num_neighbors=16, loop=True)
    edge_index_2 = safe_radius_graph(pos2, r=0.20, batch=batch2, max_num_neighbors=32, loop=True)

    assign_index_32 = safe_knn(pos3, pos2, k=3, batch_x=batch3, batch_y=batch2)
    assign_index_21 = safe_knn(pos2, pos, k=3, batch_x=batch2, batch_y=batch)

    max_n3 = int(torch.bincount(batch3).max().item())
    dummy_x = torch.zeros(pos3.size(0), 1, device=pos.device)
    _, x_dense_mask = to_dense_batch(dummy_x, batch3, max_num_nodes=max_n3)

    return (fps_idx_1, fps_idx_2, pos2, batch2, pos3, batch3,
            edge_index_1, edge_index_2, assign_index_32, assign_index_21, x_dense_mask)

# -----------------------------------------------------------------------------
# Model Components
# -----------------------------------------------------------------------------
class CliffordDiracLayer(MessagePassing):
    def __init__(self, channels: int):
        super().__init__(aggr="add", node_dim=0)
        self.channels = channels
        self.weight = nn.Linear(channels, channels, bias=False)
        self.distance_mlp = nn.Sequential(
            nn.Linear(1, channels),
            nn.SiLU(),
            nn.Linear(channels, channels),
        )
        self.resonance_mlp = nn.Sequential(
            nn.Linear(7, 16),
            nn.GELU(),
            nn.Linear(16, 1),
        )

    def forward(self, x, edge_index, v_ij, dist, edge_mask):
        x_proj = self.weight(x.transpose(1, 2)).transpose(1, 2)
        dist_weight = self.distance_mlp(dist)
        msg = self.propagate(edge_index, x=x_proj, v_ij=v_ij, dist_weight=dist_weight, edge_mask=edge_mask)
        gate = self.resonance_gate(x_proj, msg)
        return gate * msg

    def message(self, x_j, v_ij, dist_weight, edge_mask):
        v_8d = F.pad(v_ij, (1, 4)) 
        v_8d_exp = v_8d.unsqueeze(1).expand_as(x_j)
        geom_msg = triton_clifford_product(v_8d_exp, x_j)
        msg = geom_msg * dist_weight.unsqueeze(-1)
        return msg * edge_mask.view(-1, 1, 1).float()

    def resonance_gate__(self, x_state, msg_state):
        x_norm = F.normalize(x_state, dim=-1)
        msg_norm = F.normalize(msg_state, dim=-1)
        gp = triton_clifford_product(x_norm, msg_norm)

        scalar_align = gp[..., 0:1]
        vector_align = torch.norm(gp[..., 1:4], dim=-1, keepdim=True)
        bivector_align = torch.norm(gp[..., 4:7], dim=-1, keepdim=True)
        pseudoscalar_align = torch.abs(gp[..., 7:8])
        
        state_norm = torch.norm(x_state, dim=-1, keepdim=True)
        msg_norm_mag = torch.norm(msg_state, dim=-1, keepdim=True)
        delta_norm = torch.norm(msg_state - x_state, dim=-1, keepdim=True)

        feats = torch.cat([scalar_align, vector_align, bivector_align, pseudoscalar_align,
                           state_norm, msg_norm_mag, delta_norm], dim=-1)
        return torch.sigmoid(self.resonance_mlp(feats))

def resonance_gate(self, x_state, msg_state):
        # 1. KORUMA: Normalize işleminde eps değerini 1e-4'e çıkarıyoruz. 
        # FP16'da sıfıra bölme hatasını kesin engeller.
        x_norm = F.normalize(x_state, p=2, dim=-1, eps=1e-4)
        msg_norm = F.normalize(msg_state, p=2, dim=-1, eps=1e-4)
        
        gp = triton_clifford_product(x_norm, msg_norm)

        scalar_align = gp[..., 0:1]
        
        # 2. KORUMA: Norm alınan HER YERE .clamp(min=1e-4) ekliyoruz.
        # Bu sayede türev alırken x hiçbir zaman tam sıfır olmaz, gradyan patlamaz.
        vector_align = torch.norm(gp[..., 1:4], dim=-1, keepdim=True).clamp(min=1e-4)
        bivector_align = torch.norm(gp[..., 4:7], dim=-1, keepdim=True).clamp(min=1e-4)
        pseudoscalar_align = torch.abs(gp[..., 7:8])
        
        # 3. KORUMA: Durum ve mesaj normlarında da aynı güvenlik kalkanı
        state_norm = torch.norm(x_state, dim=-1, keepdim=True).clamp(min=1e-4)
        msg_norm_mag = torch.norm(msg_state, dim=-1, keepdim=True).clamp(min=1e-4)
        
        # En tehlikeli yer burasıdır! msg ve x aynı olursa delta tam sıfır olur ve patlar.
        delta_norm = torch.norm(msg_state - x_state, dim=-1, keepdim=True).clamp(min=1e-4)

        feats = torch.cat([scalar_align, vector_align, bivector_align, pseudoscalar_align,
                           state_norm, msg_norm_mag, delta_norm], dim=-1)
                           
        return torch.sigmoid(self.resonance_mlp(feats))

class CliffordSelfAttention(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.channels = channels
        self.W_q = nn.Linear(channels, channels, bias=False)
        self.W_k = nn.Linear(channels, channels, bias=False)
        self.W_v = nn.Linear(channels, channels, bias=False)
        self.score_net = nn.Sequential(nn.Linear(8, 16), nn.GELU(), nn.Linear(16, 1))

    def forward(self, x_dense, mask):
        B, N, C, _ = x_dense.shape
        q = self.W_q(x_dense.transpose(2, 3)).transpose(2, 3)
        k = self.W_k(x_dense.transpose(2, 3)).transpose(2, 3)
        v = self.W_v(x_dense.transpose(2, 3)).transpose(2, 3)
        
        q_expanded = q.unsqueeze(2)
        k_expanded = k.unsqueeze(1)
        geom_prod = triton_clifford_product(q_expanded, k_expanded)
        # Çarpımdan sonra değerlerin kontrolden çıkmasını engelle
        geom_mean = geom_prod.mean(dim=3)
        # Tanh ile sınırla (patlamayı engellemenin en sert yolu)
        scores = self.score_net(torch.tanh(geom_mean)).squeeze(-1)
        
        scores = scores.masked_fill(~(mask.unsqueeze(1) & mask.unsqueeze(2)), -10000.0)
        attn = F.softmax(scores / math.sqrt(C), dim=-1)
        return (attn.view(B, N, N, 1, 1) * v.unsqueeze(1)).sum(dim=2) + x_dense

class HierarchicalFPSCliffordNet(nn.Module):
    def __init__(self, base_channels: int, num_part_classes: int, num_categories: int):
        super().__init__()
        self.C = base_channels
        self.layer1 = CliffordDiracLayer(base_channels)
        self.layer2 = CliffordDiracLayer(base_channels * 2)
        self.lin1 = nn.Linear(base_channels * 8, (base_channels * 2) * 8)
        self.lin2 = nn.Linear((base_channels * 2) * 8, (base_channels * 4) * 8)
        self.manager = CliffordSelfAttention(base_channels * 4)
        
        # SİLİNDİ: self.cat_emb = nn.Embedding(...) (KOPYA ÇEKME İPTAL)

        # 64 birimlik o kopya vektörü çıkınca Combined Dim de küçüldü:
        combined_dim = (base_channels * 4 + base_channels * 2 + base_channels) * 8 
        
        self.head = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.BatchNorm1d(256),       
            nn.GELU(),
            nn.Dropout(0.3),           
            
            nn.Linear(256, 128),       
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),
            
            nn.Linear(128, num_part_classes)
        )

    def forward(self, pos, batch, category, fps_idx_1, fps_idx_2, pos2, batch2, pos3, batch3, 
                edge_index_1, edge_index_2, assign_index_32, assign_index_21, x_dense_mask):
        
        x0 = torch.zeros(pos.size(0), self.C, 8, device=pos.device)
        x0[..., 1:4] = pos.unsqueeze(1).expand(-1, self.C, -1)

        row1, col1 = edge_index_1[1], edge_index_1[0]
        diff1 = pos[row1] - pos[col1]
        d1 = diff1.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        dummy_mask1 = torch.ones(edge_index_1.size(1), dtype=torch.bool, device=pos.device)
        
        x1 = x0 + self.layer1(x0, edge_index_1, diff1 / d1, d1, dummy_mask1)
        f1 = (x1 / (x1.norm(dim=-1, keepdim=True).mean(dim=1, keepdim=True) + 1e-6)).reshape(x1.size(0), -1)

        x2_in = self.lin1(x1[fps_idx_1].reshape(fps_idx_1.numel(), -1)).reshape(-1, self.C * 2, 8)
        row2, col2 = edge_index_2[1], edge_index_2[0]
        diff2 = pos2[row2] - pos2[col2]
        d2 = diff2.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        dummy_mask2 = torch.ones(edge_index_2.size(1), dtype=torch.bool, device=pos.device)

        x2 = x2_in + self.layer2(x2_in, edge_index_2, diff2 / d2, d2, dummy_mask2)
        f2 = (x2 / (x2.norm(dim=-1, keepdim=True).mean(dim=1, keepdim=True) + 1e-6)).reshape(x2.size(0), -1)

        x3_in = self.lin2(x2[fps_idx_2].reshape(fps_idx_2.numel(), -1)).reshape(-1, self.C * 4, 8)
        x_dense, _ = to_dense_batch(x3_in.reshape(x3_in.size(0), -1), batch3, max_num_nodes=x_dense_mask.size(1))
        x3 = self.manager(x_dense.view(x_dense.size(0), x_dense.size(1), self.C * 4, 8), x_dense_mask)
        f3 = x3[x_dense_mask].reshape(-1, self.C * 4 * 8)

        row_32, col_32 = assign_index_32[0], assign_index_32[1]
        out_f3_to_pos2 = torch.zeros(pos2.size(0), f3.size(1), device=pos.device)
        out_f3_to_pos2.scatter_add_(0, row_32.unsqueeze(1).expand(-1, f3.size(1)), f3[col_32])
        out_f3_to_pos2 = out_f3_to_pos2 / 3.0
        
        f2_combined = torch.cat([out_f3_to_pos2, f2], dim=-1)
        
        row_21, col_21 = assign_index_21[0], assign_index_21[1]
        f2_up = torch.zeros(pos.size(0), f2_combined.size(1), device=pos.device)
        f2_up.scatter_add_(0, row_21.unsqueeze(1).expand(-1, f2_combined.size(1)), f2_combined[col_21])
        f2_up = f2_up / 3.0
        
        # SİLİNDİ: cat_features = self.cat_emb(category)[batch]
        # SADECE KENDİ ÖĞRENDİKLERİYLE (f2_up ve f1) BİRLEŞTİR:
        f1_final = torch.cat([f2_up, f1], dim=-1)

        return self.head(f1_final)

# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------

def compute_shape_miou(pred: torch.Tensor, target: torch.Tensor, valid_parts: list) -> float:
    ious = []
    for cls in valid_parts:
        # DÜZELTME: 0 numaralı sınıf (Arka plan / Etiketsiz alan) metriğe dahil EDİLEMEZ!
        if cls == 0:
            continue
            
        pred_mask, target_mask = (pred == cls), (target == cls)
        union = (pred_mask | target_mask).sum().item()
        
        if union == 0: 
            # Hedefte bu sınıf var ama hiç kesişim/birleşim olmadıysa skor 0'dır.
            ious.append(0.0)
        else:
            ious.append((pred_mask & target_mask).sum().item() / union)
            
    # Eğer nesne tamamen etiketsizse (sadece 0'lardan oluşuyorsa) skoru etkilemesin
    if len(ious) == 0:
        return float('nan') 
        
    return float(np.mean(ious))

# Bu fonksiyonu SİLİN VEYA İÇİNİ BOŞALTIN (Hiçbir şeyi maskelemesin)
def get_masked_logits(logits: torch.Tensor, category: torch.Tensor, batch: torch.Tensor, cat_to_parts: dict) -> torch.Tensor:
    # Artık kategoriye göre sınırlandırma (maskeleme) YAPMIYORUZ.
    # Model tüm 315 sınıftan istediğini seçmekte özgür!
    return logits

@torch.no_grad()
def evaluate(model, loader, device, criterion, cat_to_parts, cat_names):
    model.eval()
    total_loss = 0.0
    
    
    shape_ious = []
    # category_ious = defaultdict(list)  <-- Bunu kaldırıyoruz
    
    for data in loader:
        data = data.to(device)
        
        (fps_idx_1, fps_idx_2, pos2, batch2, pos3, batch3, 
         edge_index_1, edge_index_2, assign_index_32, assign_index_21, x_dense_mask) = prepare_batch_data(data)

        logits = model(data.pos, data.batch, data.category, fps_idx_1, fps_idx_2, pos2, batch2, pos3, batch3,
                       edge_index_1, edge_index_2, assign_index_32, assign_index_21, x_dense_mask)
        
        # main'den gelen FocalLoss burada çalışacak:
        loss = criterion(logits, data.y) 
        total_loss += loss.item() * data.num_graphs

        # Maskeleme yok, doğrudan en yüksek ihtimalli tahmini al
        pred = logits.argmax(dim=-1)

        # (evaluate içindeki döngü)
        for i in range(data.num_graphs):
            mask = data.batch == i
            
            target_labels = data.y[mask]
            valid_parts_in_object = torch.unique(target_labels).tolist()
            
            iou = compute_shape_miou(pred[mask], target_labels, valid_parts_in_object)
            if not np.isnan(iou):  # DÜZELTME: nan dönenleri listeye alma
                shape_ious.append(iou)

    inst_miou = float(np.mean(shape_ious))

    print("\n" + "="*50)
    print(f"SOTA mIoU (Class-Agnostic) : {inst_miou * 100:.2f} | Toplam Test Edilen Nesne: {len(shape_ious)}")
    print("="*50 + "\n")

    return total_loss / len(loader.dataset), inst_miou, inst_miou # class_miou yerine inst_miou döndür
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', ignore_index=0, label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets.data.view(-1))
            focal_loss = focal_loss * alpha_t
        return focal_loss.mean()

# -----------------------------------------------------------------------------
# Training Loop
# -----------------------------------------------------------------------------
@dataclass
class Config:
    cache_root: str = "./data/PartNeXt"       # Tensor dosyalarının kaydedileceği yer
    mesh_root: str = "/data/PartNeXt/mesh_root/glbs"       # İndirdiğiniz 3D nesnelerin bulunduğu yer
    num_points: int = 2048
    batch_size: int = 16         # Nokta sayısı arttığı için VRAM patlamaması adına düşürüldü
    lr: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 50
    base_channels: int = 32       # Kapasite artırıldı (PartNeXt daha detaylı)
    seed: int = 42
    force_rebuild_cache: bool = False
    save_path: str = "./best_partnext_clifford.pt"

from torch.utils.data import random_split
from tqdm import tqdm
def main():
    cfg = Config()
    seed_everything(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    cat_names, cat_to_parts, num_categories, total_parts = load_partnext_mappings(cfg.cache_root)
    print(f"Toplam Kategori: {num_categories}, Toplam Parça Sınıfı: {total_parts}")

    # 1. HuggingFace'de sadece 'train' olduğu için TÜM VERİYİ tek seferde çekiyoruz
    full_ds = materialize_split(cfg.cache_root, cfg.mesh_root, "train", cfg.num_points, cfg.seed, cfg.force_rebuild_cache)
    
    # 2. Veriyi manuel olarak %80 Train, %20 Test olacak şekilde bölüyoruz
    total_size = len(full_ds)
    train_size = int(0.8 * total_size)
    test_size = total_size - train_size
    
    print(f"Veri Seti Bölünüyor: {train_size} Eğitim, {test_size} Test")
    
    # Sabit bir seed veriyoruz ki her çalıştırmada aynı modeller teste düşsün, sonuçlar tutarlı olsun
    generator = torch.Generator().manual_seed(cfg.seed)
    train_ds, test_ds = random_split(full_ds, [train_size, test_size], generator=generator)

    # 3. DataLoader'lara yeni böldüğümüz veri setlerini veriyoruz
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, 
                              num_workers=4, pin_memory=True, persistent_workers=True, drop_last=True)
    
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, 
                             num_workers=4, pin_memory=True, persistent_workers=True)

    # ... Modeli oluşturma ve Eğitime (Epoch döngüsü) başlama kısmı aynı kalacak ...
    model = HierarchicalFPSCliffordNet(
        base_channels=cfg.base_channels, 
        num_part_classes=total_parts,
        num_categories=num_categories
    ).to(device)
    
    
    
    model = model.to(device)

    if os.path.exists(cfg.save_path):
        try:
            print(f"[{time.strftime('%H:%M:%S')}] Mevcut en iyi ağırlıklar yükleniyor...")
            
            # 1. Ağırlıkları belleğe al
            checkpoint = torch.load(cfg.save_path, map_location=device, weights_only=True)
            
            # 2. Temiz bir sözlük oluştur
            clean_state_dict = {}
            for key, value in checkpoint.items():
                # 'module.' önekini kaldır, diğer kısımları (örn: _orig_mod) önceki gibi temizle
                clean_key = key.replace("module.", "", 1) if key.startswith("module.") else key
                clean_key = clean_key.replace("_orig_mod.", "") if clean_key.startswith("_orig_mod.") else clean_key
                
                clean_state_dict[clean_key] = value
                
            # 3. Temizlenmiş sözlüğü modele yükle
            model.load_state_dict(clean_state_dict, strict=True)
            print("Ağırlıklar başarıyla yüklendi!")
            
        except Exception as e:
            print(f"Ağırlıklar yüklenirken hata oluştu: {e}. Sıfırdan başlanıyor.")

    try:
        model = torch.compile(model)
    except Exception as e:
        print(f"torch.compile hatası: {e}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    #criterion = FocalLoss(gamma=2.0, label_smoothing=0.0)
    criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)
    scaler = GradScaler('cuda')
    
    best_inst_miou = -1.0 

    for epoch in range(1, cfg.epochs + 1):
        t0 = time.time()
        model.train()
        total_train_loss = 0.0

        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad(set_to_none=True) 
            
            (fps_idx_1, fps_idx_2, pos2, batch2, pos3, batch3, 
             edge_index_1, edge_index_2, assign_index_32, assign_index_21, x_dense_mask) = prepare_batch_data(data)

            with autocast('cuda'):
                raw_logits = model(data.pos, data.batch, data.category, 
                                   fps_idx_1, fps_idx_2, pos2, batch2, pos3, batch3,
                                   edge_index_1, edge_index_2, assign_index_32, assign_index_21, x_dense_mask)
                
                masked_logits = get_masked_logits(raw_logits, data.category, data.batch, cat_to_parts)
                loss = criterion(masked_logits, data.y)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
            scaler.step(optimizer)
            scaler.update()
            
            total_train_loss += loss.item() * data.num_graphs

        train_loss = total_train_loss / len(train_loader.dataset)
        val_loss, val_inst_miou, _ = evaluate(model, test_loader, device, criterion, cat_to_parts, cat_names)
        
        scheduler.step(val_inst_miou)
        current_lr = optimizer.param_groups[0]['lr']

        if val_inst_miou > best_inst_miou:
            best_inst_miou = val_inst_miou
            torch.save(model.state_dict(), cfg.save_path)
            mark = "*"
        else:
            mark = " "

        print(f"Epoch {epoch:02d} | time {time.time()-t0:.1f}s | lr: {current_lr:.2e} | loss: {train_loss:.4f} | "
              f"Val Inst. mIoU: {val_inst_miou*100:.2f} {mark}")
        
if __name__ == "__main__":
    import torch_cluster
    print("Torch Cluster CUDA destekli mi?:", torch_cluster.cuda_version)
    main()

Torch Cluster CUDA destekli mi?: 12080
Kategori haritaları önbellekten (cache) yükleniyor...
Toplam Kategori: 248, Toplam Parça Sınıfı: 315
[09:52:12] Cache bulundu: ./data/PartNeXt/partnext_train_n2048_seed42_fps.pt
Veri Seti Bölünüyor: 18815 Eğitim, 4704 Test
[09:52:21] Mevcut en iyi ağırlıklar yükleniyor...
Ağırlıklar yüklenirken hata oluştu: Error(s) in loading state_dict for HierarchicalFPSCliffordNet:
	size mismatch for layer1.weight.weight: copying a param with shape torch.Size([16, 16]) from checkpoint, the shape in current model is torch.Size([32, 32]).
	size mismatch for layer1.distance_mlp.0.weight: copying a param with shape torch.Size([16, 1]) from checkpoint, the shape in current model is torch.Size([32, 1]).
	size mismatch for layer1.distance_mlp.0.bias: copying a param with shape torch.Size([16]) from checkpoint, the shape in current model is torch.Size([32]).
	size mismatch for layer1.distance_mlp.2.weight: copying a param with shape torch.Size([16, 16]) from checkpoin

W0428 09:52:31.664000 1898 torch/_inductor/utils.py:1679] [0/0_1] Not enough SMs to use max_autotune_gemm mode



SOTA mIoU (Class-Agnostic) : 25.88 | Toplam Test Edilen Nesne: 3061

Epoch 01 | time 625.7s | lr: 1.00e-03 | loss: 2.9868 | Val Inst. mIoU: 25.88 *

SOTA mIoU (Class-Agnostic) : 28.62 | Toplam Test Edilen Nesne: 3061

Epoch 02 | time 547.0s | lr: 1.00e-03 | loss: 2.6467 | Val Inst. mIoU: 28.62 *

SOTA mIoU (Class-Agnostic) : 29.74 | Toplam Test Edilen Nesne: 3061

Epoch 03 | time 546.2s | lr: 1.00e-03 | loss: 2.5654 | Val Inst. mIoU: 29.74 *

SOTA mIoU (Class-Agnostic) : 30.66 | Toplam Test Edilen Nesne: 3061

Epoch 04 | time 546.2s | lr: 1.00e-03 | loss: 2.5075 | Val Inst. mIoU: 30.66 *

SOTA mIoU (Class-Agnostic) : 31.87 | Toplam Test Edilen Nesne: 3061

Epoch 05 | time 545.5s | lr: 1.00e-03 | loss: 2.4734 | Val Inst. mIoU: 31.87 *

SOTA mIoU (Class-Agnostic) : 31.71 | Toplam Test Edilen Nesne: 3061

Epoch 06 | time 544.9s | lr: 1.00e-03 | loss: 2.4395 | Val Inst. mIoU: 31.71  

SOTA mIoU (Class-Agnostic) : 31.15 | Toplam Test Edilen Nesne: 3061

Epoch 07 | time 543.6s | lr: 1.00e-0

KeyboardInterrupt: 